# bias-correction-divide — worked example 1: Bias-correct the Adam second-moment buffer v at a given step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bias-correction-divide`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Adam keeps two EMA buffers: the first moment `m` (decay `beta1`) and the second moment `v` (decay `beta2`). Both start at zero, so both are biased toward zero early in training. The fix is the same divide: `v_hat = v / (1 - beta2 ** t)`. The 1-based step `t` lives in the exponent, so at `t=1` the correction is the largest (`1/(1-beta2)`) and shrinks toward 1 as training proceeds.

## Worked solution

**Goal:** given a raw second-moment buffer `v`, the decay `beta2`, and the 1-based step `t`, return the unbiased `v_hat`.

**Step 1 — compute the correction denominator.** `1 - beta2 ** t` is a plain Python scalar. With `beta2=0.999` and `t=1` this is `0.001`, so dividing by it multiplies `v` by `1000` — exactly the inflation needed to undo the zero-init bias of a one-step EMA.

**Step 2 — divide, don't mutate.** `v / correction` returns a NEW tensor; we never write back into `v` in place, because the optimizer still needs the raw EMA buffer for the next recurrence step. Mutating `v` would corrupt the running average.

**Step 3 — why it is correct.** A zero-initialized EMA after `t` constant updates equals `(1 - beta2**t)` times the target. Dividing by exactly that factor recovers the target, which is why a constant input yields a flat corrected output.

In [ ]:
def bias_correct_v(v, beta2, t_step):
    correction = 1 - beta2 ** t_step
    return v / correction

t.manual_seed(0)
v = t.tensor([0.002, 0.004, 0.006])
v_hat = bias_correct_v(v, 0.999, 2)
print("v     :", v)
print("v_hat :", v_hat)
print("factor:", 1 / (1 - 0.999 ** 2))